In [31]:
import os
import scipy.io as sc
import numpy as np

In [32]:
def extract_label(folder, filename) :
    label_map = {
        'normal': 0, 'B007': 1, 'B014': 2, 'B021': 3,  
        'IR007': 4, 'IR014': 5, 'IR021': 6, 
        'OR007': 7, 'OR014': 8, 'OR021': 9  
    }
    if 'normal' in folder.lower():
        return 0
    name = filename.split('_')[0]
    for key in label_map:
        if key in name:
            return label_map[key]
    return -1;

In [33]:
def split_signal(signal):
    length = len(signal)

    train_end = int(length * 0.70)
    val_end = int(length * 0.85)

    s_train = signal[:train_end]
    s_val = signal[train_end:val_end]
    s_test = signal[val_end:]

    return s_train, s_val, s_test

In [49]:
def split_block(block, label, wnd_size=2048):
    no_windows = len(block) // wnd_size
    if no_windows == 0:
        return None, None
        
    trimmed = block[: no_windows * wnd_size]
    x_matrix = trimmed.reshape(no_windows, wnd_size)
    y_matrix = np.full(no_windows, label, dtype=np.int64)
    
    return x_matrix, y_matrix
    
    

In [35]:
def extract_signal(root_path, output_dir="..\\processed_data"):
    X_tr_list, y_tr_list = [], []
    X_v_list, y_v_list = [], []
    X_te_list, y_te_list = [], []


    for root, dirs, files in os.walk(root_path):
        for file in files:
            if file.endswith('.mat'):
                file_path = os.path.join(root, file)
                label = extract_label(root,file)

                if label == -1 :
                    continue;
                load_sig = sc.loadmat(file_path)
                mat_key = None
                for key in load_sig.keys():
                    if "_DE_time" in key:
                        mat_key = key
                        break
                        
                if mat_key is None:
                    continue
                signal = load_sig[mat_key].flatten()
                s_train, s_val, s_test = split_signal(signal)

                x_tr, y_tr = split_block(s_train, label, wnd_size = 2048)
                x_v, y_v = split_block(s_val, label, wnd_size = 2048)
                x_te, y_te = split_block(s_test, label, wnd_size = 2048)

                if x_tr is not None:
                    X_tr_list.append(x_tr)
                    y_tr_list.append(y_tr)
                if x_v is not None:
                    X_v_list.append(x_v)
                    y_v_list.append(y_v)
                if x_te is not None:
                    X_te_list.append(x_te)
                    y_te_list.append(y_te)
                
    os.makedirs(output_dir, exist_ok=True)
    
    np.save(os.path.join(output_dir, "X_train.npy"), np.vstack(X_tr_list))
    np.save(os.path.join(output_dir, "y_train.npy"), np.concatenate(y_tr_list))
    
    np.save(os.path.join(output_dir, "X_val.npy"), np.vstack(X_v_list))
    np.save(os.path.join(output_dir, "y_val.npy"), np.concatenate(y_v_list))
    
    np.save(os.path.join(output_dir, "X_test.npy"), np.vstack(X_te_list))
    np.save(os.path.join(output_dir, "y_test.npy"), np.concatenate(y_te_list))           

In [50]:
extract_signal(root_path="../../Data")